In [2]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
documents = [
    "Ahmed is a third-year AI and Data Science student.",
    "He is learning Retrieval-Augmented Generation to build a dental lecture assistant.",
    "A convolutional neural network is commonly used for image classification tasks.",
    "Passive eruption is related to the movement of the gingival margin after tooth eruption.",
    "Power BI is used to build dashboards and visualize business data."
]

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
doc_embeddings = model.encode(documents)
print(type(doc_embeddings))
print(doc_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [6]:
question = "What Project is Ahmed building with RAG"

question_embedding = model.encode(question)
print(question_embedding.shape)

(384,)


In [7]:
def cosine_similarity(a, b):
    dot_product = np.dot(a,b)
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    return dot_product / (mag_a * mag_b)

In [20]:
similarties = []

for doc, emb in zip(documents, doc_embeddings):
    similarity = cosine_similarity(emb, question_embedding)
    similarties.append(similarity)

result = pd.DataFrame({"document": documents, "similarity": similarties})

result = result.sort_values(by="similarity", ascending=True)
result

,document,similarity
3,Passive eruption is related to the movement of...,-0.013756
2,A convolutional neural network is commonly use...,0.030239
1,He is learning Retrieval-Augmented Generation ...,0.148563
4,Power BI is used to build dashboards and visua...,0.192862
0,Ahmed is a third-year AI and Data Science stud...,0.365891


In [62]:
def retrieve_top_k(question, documents, doc_embeddings, k=3):
    question_embedding = model.encode(question)

    scores = []
    for emb in doc_embeddings:
        score = cosine_similarity(question_embedding, emb)
        scores.append(score)

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "document": documents[idx],
            "score": scores[idx]
        })

    return retrieved

In [63]:
retrieve_top_k(question, documents, doc_embeddings)

[{'document': 'Ahmed is a third-year AI and Data Science student.',
  'score': np.float32(0.36589068)},
 {'document': 'Power BI is used to build dashboards and visualize business data.',
  'score': np.float32(0.19286232)},
 {'document': 'He is learning Retrieval-Augmented Generation to build a dental lecture assistant.',
  'score': np.float32(0.14856318)}]